# Learnings
- `Q.backward()` only works when Q is a scalar, not a vector, you need to pass a vector as graident to `Q.backward(gradient=torch.tensor([1., 1.]))` when Q is a 2d vector for example, to signify its gradient w.r.t itself
  - Or you could do `Q.sum().backward()` => Would lead to the same a and b grad values
- So in classification problems, output of final layer or the NN is nn.Linear(512, 10) which are logits of size 10.
  - when you do CELoss, the softmax is done within the loss calculation, so you dont need a softmax in the NN
  - However at test time, say there's a batch of 4, the labels are like `[0, 3, 4, 1]`, then outputs are of shape `[4, 10]`, you will need to do `torch.argmax(outputs, dim=1)` across the column dimension to get the index and then do calculation of correct v/s not correct.
  - Since the max value wont change before or after applying softmax, it is safe to take the argmax

In [ ]:
# https://docs.pytorch.org/tutorials/beginner/deep_learning_60min_blitz.html

In [ ]:
# Autograd (once again)

In [ ]:
import torch

In [ ]:
a = torch.tensor([2., 3.], requires_grad=True)
b = torch.tensor([6., 4.], requires_grad=True)

In [ ]:
a.shape, b.shape

(torch.Size([2]), torch.Size([2]))

In [ ]:
Q = 3 * a.pow(3) - b.pow(2)

In [ ]:
Q

tensor([-12.,  65.], grad_fn=<SubBackward0>)

In [ ]:
Q.backward(gradient=torch.tensor([1., 1.]))

In [ ]:
a.grad

tensor([36., 81.])

In [ ]:
b.grad

tensor([-12.,  -8.])

In [ ]:
a.grad, b.grad = None, None
Q_new = 3 * a.pow(3) - b.pow(2)

In [ ]:
Q_new.sum().backward()

In [ ]:
a.grad

tensor([36., 81.])

In [ ]:
b.grad

tensor([-12.,  -8.])

In [ ]:
# check if collected gradients are correct
print(9*a**2 == a.grad)
print(-2*b == b.grad)

tensor([True, True])
tensor([True, True])


In [ ]:
# Optim

In [ ]:
model = torch.nn.Sequential(
    torch.nn.Linear(10, 5),
    torch.nn.ReLU(),
    torch.nn.Linear(5, 2),
    torch.nn.ReLU()
)

In [ ]:
x = torch.rand(100, 10)
y_pred = model(x)

In [ ]:
x.shape, y.shape

(torch.Size([100, 10]), torch.Size([100, 2]))

In [ ]:
y = torch.rand(100, 2)

In [ ]:
loss = torch.nn.MSELoss()(y_pred, y)

In [ ]:
loss.backward()

In [ ]:
lr = 1e-6
for param in model.parameters():
  param.data = param.data - lr * param.grad.data

# OR

lr = 1e-6
for param in model.parameters():
  param.data.sub_(lr * param.grad.data)

In [ ]:
# Classifier

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms

In [ ]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 4

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

100%|██████████| 170M/170M [00:03<00:00, 48.8MB/s]


In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net()

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [ ]:
net.to('cuda')
for epoch in range(2):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data
        inputs = inputs.to('cuda')
        labels = labels.to('cuda')

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

print('Finished Training')

[1,  2000] loss: 2.198
[1,  4000] loss: 1.854
[1,  6000] loss: 1.656
[1,  8000] loss: 1.558
[1, 10000] loss: 1.477
[1, 12000] loss: 1.434
[2,  2000] loss: 1.369
[2,  4000] loss: 1.340
[2,  6000] loss: 1.325
[2,  8000] loss: 1.280
[2, 10000] loss: 1.293
[2, 12000] loss: 1.248
Finished Training


In [ ]:
inputs.shape

torch.Size([4, 3, 32, 32])

In [ ]:
labels.shape

torch.Size([4])

In [ ]:
labels

tensor([9, 8, 5, 9], device='cuda:0')

In [ ]:
outputs.shape

torch.Size([4, 10])

In [ ]:
outputs[0]

tensor([ 0.1453,  1.8957, -1.5407,  0.1536, -0.3432, -0.3250, -2.1038,  0.0359,
        -1.5991,  3.7910], device='cuda:0', grad_fn=<SelectBackward0>)

In [ ]:
_, predicted = torch.max(outputs, 1)

In [ ]:
predicted

tensor([9, 8, 7, 9], device='cuda:0')

In [ ]:
## Test loop
net.eval()
correct, total = 0, 0
with torch.no_grad():
  for data in testloader:
    images, labels = data
    images = images.to('cuda')
    labels = labels.to('cuda')
    outputs = net(images)
    predicted = torch.argmax(outputs, dim=1)
    total += len(predicted)
    correct += (predicted == labels).sum().item()

In [ ]:
correct/total

0.5565

Net(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

In [ ]:
# Next (in new notebook):  https://docs.pytorch.org/tutorials/beginner/nn_tutorial.html